# MiniMax H3 Max

<p align="center">
  <img src="../../.github/images/banners/fal-minimax-h3-max.png" alt="MiniMax H3 Max: a weekend trip with our real products, one scene, one soundtrack" width="900" />
</p>

Text-, image-, and reference-to-video with first, middle, and last frames, prompt expansion, and a soundtrack we can pin, from MiniMax (post-trained by fal) on Microsoft Foundry.

| | |
|---|---|
| Released | 2026-11-06 (preview) |
| Publisher | fal (model by MiniMax, post-trained by fal) |
| Capability | [Video Generation](../../3-primers/video-generation.md) |
| Model card | [Fal-MiniMax-H3-Max](https://ai.azure.com/catalog/models/Fal-MiniMax-H3-Max) |
| Announcement | [Release post](https://techcommunity.microsoft.com/blog/azure-ai-foundry-blog/building-the-next-generation-of-multimodal-ai-experiences-with-fal-in-microsoft-/4570032) |
| Docs | [Fal models on Microsoft Foundry](https://learn.microsoft.com/azure/foundry/how-to/fal-ai-models/enable-fal-models) |
| Deployment | Global Standard, pay-as-you-go |
| Modes | Text-to-video, image-to-video (first and/or last frame), **reference-to-video** (up to 12 files: 9 images, 3 videos, 3 audio clips) |
| Clip shape | 0.92–15 s (default 5; output can run ~0.7 s longer) · 480P, 768P, 1080P (refined from 768P) · 21:9 to 9:16 · audio always generated |
| Rate limit | 2 requests per minute on quota tiers 1–4 |
| Pricing | Not on the model card yet. Check your deployment in the Foundry portal; fal's direct price is on [fal.ai](https://fal.ai/models/minimax/h3-max/text-to-video) |
| Env vars | `MICROSOFT_FOUNDRY_ENDPOINT`, `AZURE_FAL_MINIMAX_H3_MAX_DEPLOYMENT`, `DEMO_MODE` |

**Read first:** the [Video Generation primer](../../3-primers/video-generation.md) explains the terms we use here (reference-to-video, first and last frame, prompt expansion, seed, native audio). Ten minutes there makes this notebook much easier.

**First time here?** Run [setup.ipynb](../../setup.ipynb) once to create your Microsoft Foundry project and `.env`.

> *"Can MiniMax H3 Max put our real products, a person, and our own soundtrack into one scene, and do what the prompt says?"*

Contoso Outdoors is making a creator-style "weekend trip" campaign: a hiker, the TrailMaster X4 tent, the Adventurer Pro backpack, and the EcoFire stove, all recognizable, all in one clip, cut to our own voiceover. H3 Max is the quality tier; its sibling [MiniMax H3 Max Turbo](../fal-minimax-h3-max-turbo/) is faster and cheaper. We use the [Contoso Outdoors scenario](../../2-scenarios/contoso-retail/) and launch images made by an earlier MAI workshop.

## What We Will Do

- See what prompt expansion does to our words, with a fixed seed for fair comparisons
- Put four reference images into one scene, and steer a clip with first, middle, and last frames
- Pin our own voiceover as the soundtrack, and hold a visual style
- Check 768P vs 1080P and requested vs delivered duration, then package it for a customer conversation

> Run each cell with **Shift + Enter**. Guess first: before each video, predict one thing it will get right and one it will get wrong.

<img src="../../.github/images/divider.svg" alt="" width="100%" height="6" />

## Before you start

**Just reading?** Leave `DEMO_MODE=replay`. The notebook shows clips saved in `examples/` and makes no Azure calls.

**Running it live?** We need four things:

1. **fal enabled on the subscription.** An admin registers the **Fal.EnableDeploy** preview feature ([steps](https://learn.microsoft.com/azure/foundry/how-to/fal-ai-models/enable-fal-models#enable-fal-on-foundry)). It can take 30 minutes.
2. **A MiniMax H3 Max deployment** (Global Standard) in our Foundry project ([steps](https://learn.microsoft.com/azure/foundry/how-to/fal-ai-models/enable-fal-models#deploy-fal-models-from-the-foundry-portal)).
3. **A sign-in with a Foundry role** on the resource: run `az login` in a terminal. We use keyless Microsoft Entra ID tokens, never API keys.
4. **`.env` at the repo root** with `MICROSOFT_FOUNDRY_ENDPOINT`, `AZURE_FAL_MINIMAX_H3_MAX_DEPLOYMENT`, and `DEMO_MODE=live`. See [sample.env](../../1-scripts/sample.env).

**Data handling.** fal hosts this model outside Microsoft. Prompts, images, and audio we send leave Microsoft, and Microsoft's product terms and DPA don't apply. We only send fictional Contoso data and generated images here, never photos of real people.

In [ ]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "1-scripts" / "fal_video.py").exists())
load_dotenv(ROOT / ".env")

DEMO_MODE = None  # Set to "live" or "replay" to override .env
MODE = (DEMO_MODE or os.getenv("DEMO_MODE") or "replay").strip().lower()
ENDPOINT = os.getenv("MICROSOFT_FOUNDRY_ENDPOINT", "").strip()
DEPLOYMENT = os.getenv("AZURE_FAL_MINIMAX_H3_MAX_DEPLOYMENT", "Fal-MiniMax-H3-Max").strip()
RPM = int(os.getenv("FAL_REQUESTS_PER_MINUTE", "2"))
PLAY_INLINE = os.getenv("PLAY_INLINE", "true").strip().lower() == "true"

if MODE not in ("replay", "live"):
    raise SystemExit(f'DEMO_MODE is "{MODE}". Use "replay" or "live".')

credential = None
if MODE == "live":
    if not ENDPOINT.startswith("https://"):
        raise SystemExit("MICROSOFT_FOUNDRY_ENDPOINT is missing from .env. Run setup.ipynb, or copy it from the project's Overview page.")
    from azure.identity import DefaultAzureCredential

    credential = DefaultAzureCredential()
    try:
        credential.get_token("https://cognitiveservices.azure.com/.default")
    except Exception as error:
        raise SystemExit(f"Couldn't get a Microsoft Entra token. Run `az login` in a terminal, then run this cell again.\n{error}")

print(f"Mode: {MODE} | Deployment: {DEPLOYMENT} | {RPM} requests per minute")

The shared helper in [1-scripts/fal_video.py](../../1-scripts/fal_video.py) submits each job, polls until it's done, downloads the MP4, and saves a JSON record, including two fields MiniMax returns that other models don't: `expanded_prompt` and `timings`. `expanded()` reads the first one back out.

In [ ]:
sys.path.insert(0, str(ROOT / "1-scripts"))
from fal_video import FalVideo, as_url, scoreboard

HERE = ROOT / "4-models" / "fal-minimax-h3-max"
GENERATED = ROOT / "2-scenarios" / "contoso-retail" / "data" / "generated"
IMAGES = GENERATED / "images"
VOICEOVER = GENERATED / "audio" / "trailmaster-x4-voiceover.mp3"

fal = FalVideo(ENDPOINT, DEPLOYMENT, family="minimax/h3-max", here=HERE, mode=MODE,
               requests_per_minute=RPM, play_inline=PLAY_INLINE, credential=credential)
clips = []
SEED = 2026  # One seed for every comparison, so differences come from the setting we change


def expanded(clip) -> str:
    return clip.record.get("response", {}).get("expanded_prompt") or "(none: expansion was off or left the prompt unchanged)"

<img src="../../.github/images/divider.svg" alt="" width="100%" height="6" />

## 1. What does the model actually receive?

H3 Max rewrites prompts before generating (`prompt_expansion_mode`): `disabled`, `balanced` (about a second, the default), or `quality` (up to ~30 s). We send one short prompt three ways with the same seed, then **read** what each mode sent to the model.

> 🔮 **Guess first:** will the short prompt or the expanded one look more like a real ad?

In [ ]:
SHORT = "A hiker sets up a green and orange tent by a lake at sunset."

modes = {}
for expansion in ("disabled", "balanced", "quality"):
    modes[expansion] = fal.run(f"01-expansion-{expansion}", "text-to-video", prompt=SHORT, prompt_expansion_mode=expansion,
                               seed=SEED, duration=5, resolution="768P", aspect_ratio="16:9")
    print(f"\n{expansion.upper()}: {expanded(modes[expansion])}\n")
clips += modes.values()
fal.show(*modes.values())

**What to look for:** what the expansion **added**: camera moves, light, sound, details we never asked for. Expansion helps short prompts and can override careful ones. For brand work, write the full shot yourself and set `disabled`. For exploration, `balanced` is a cheap idea generator: read its output and keep the parts you like.

<img src="../../.github/images/divider.svg" alt="" width="100%" height="6" />

## 2. Can four real products share one scene?

**Reference-to-video** takes images the clip must stay faithful to, without fixing any single frame. The prompt names them by position: **Image 1**, **Image 2**, and so on. Here: the hiker, the backpack, the tent, and the stove from the earlier workshop.

In [ ]:
from IPython.display import Image as Picture, display

CAST = ["prop-hiker", "prop-backpack", "prop-tent", "prop-stove"]
display(*[Picture(filename=str(IMAGES / f"{name}.webp"), width=150) for name in CAST])

WEEKEND = """The hiker from Image 1, wearing the backpack from Image 2, arrives at a lakeside campsite in late afternoon.
The hiker sets down the pack beside the tent from Image 3, already pitched, and lights the stove from Image 4; a small pot starts to steam.
Medium tracking shot that ends on a close-up of the stove. Warm light.
Audio: footsteps on gravel, the click and hiss of the stove, birds."""

weekend = fal.run("02-reference-four-products", "reference-to-video", prompt=WEEKEND, prompt_expansion_mode="disabled",
                  reference_image_urls=[as_url(IMAGES / f"{name}.webp") for name in CAST],
                  seed=SEED, duration=8, resolution="768P", aspect_ratio="16:9")
clips.append(weekend)
fal.show(weekend, frames=6)

**What to look for:** go product by product. The tent's green body and orange rainfly? The backpack's color and straps? The stove's shape? Then check the **order of events**: arrive, set down, light. H3 Max is tuned to hit beats in the order written. Any product that drifts is a candidate for its own start frame.

<img src="../../.github/images/divider.svg" alt="" width="100%" height="6" />

## 3. Can we steer with first, middle, and last frames?

Three controls, from loose to tight:

- **End frame only:** the clip *lands* on our image. We land on the launch poster.
- **First and last:** we fix both ends; the model invents the middle.
- **First, middle, last:** a third image at a set time (reference-to-video route, 480P or 768P only).

In [ ]:
display(Picture(filename=str(IMAGES / "launch-poster.webp"), width=180),
        Picture(filename=str(IMAGES / "campsite-before.webp"), width=240),
        Picture(filename=str(IMAGES / "lifestyle-all-products.webp"), width=180),
        Picture(filename=str(IMAGES / "campsite-after.webp"), width=240))

land = fal.run("03-end-frame-only", "image-to-video", end_image_url=as_url(IMAGES / "launch-poster.webp"), prompt_expansion_mode="disabled",
               prompt="Fog clears over an alpine lake at dawn, revealing the tent on a granite ledge; the scene settles into the final poster.",
               seed=SEED, duration=5, resolution="768P")

bookends = fal.run("03-first-and-last", "image-to-video", image_url=as_url(IMAGES / "campsite-before.webp"),
                   end_image_url=as_url(IMAGES / "campsite-after.webp"), prompt_expansion_mode="disabled",
                   prompt="A gust of wind crosses the lake; as it passes, the tent's rainfly changes to its new color. Static camera.",
                   seed=SEED, duration=5, resolution="768P")

three_frames = fal.run("03-first-middle-last", "reference-to-video", image_url=as_url(IMAGES / "campsite-before.webp"),
                       middle_image_url=as_url(IMAGES / "lifestyle-all-products.webp"), middle_frame_time=3.0,
                       end_image_url=as_url(IMAGES / "campsite-after.webp"), prompt_expansion_mode="disabled",
                       prompt="Morning at the lakeside camp; the camera moves in to the hiker unpacking by the stove, then pulls back to the tent in its new color.",
                       seed=SEED, duration=6, resolution="768P")
clips += [land, bookends, three_frames]
fal.show(land, bookends, three_frames, frames=6)

**What to look for:** the last frame of the first clip should *be* the poster, text and all; the first and last frames of the other two should match their images. In the three-frame clip, find the middle image at about 3 seconds. Note the canvas: with an end image only, the output takes the end image's shape (vertical here).

<img src="../../.github/images/divider.svg" alt="" width="100%" height="6" />

## 4. Can we use our own soundtrack?

`target_audio_url` **pins** an audio clip (at least 2 seconds, up to 15 MB) as the soundtrack: the model generates the picture to go with it, instead of inventing a soundtrack. Our clip is a short TrailMaster X4 voiceover made with [MAI-Voice-2](../mai-voice-2/), sent as a data URI.

In [ ]:
from IPython.display import Audio

if VOICEOVER.exists():
    display(Audio(filename=str(VOICEOVER)))
    pinned = fal.run("04-pinned-voiceover", "image-to-video", image_url=as_url(IMAGES / "lifestyle-all-products.webp"),
                     target_audio_url=as_url(VOICEOVER), prompt_expansion_mode="disabled",
                     prompt="The hiker relaxes at camp as the stove steams and the light turns gold; slow push in toward the tent.",
                     seed=SEED, duration=8, resolution="768P")
    clips.append(pinned)
    fal.show(pinned)
else:
    print(f"No voiceover at {VOICEOVER.relative_to(ROOT)} yet. See the Contoso Outdoors scenario README for how it's made.")

**What to look for:** the clip's audio should be **our** voiceover, word for word, resampled but not sped up. If the voiceover is shorter than the clip, the end is silent; if it's longer, it's trimmed. Pinned audio can't be combined with reference audio in the same request.

<img src="../../.github/images/divider.svg" alt="" width="100%" height="6" />

## 5. Can it hold a visual style?

H3 Max is tuned to keep a consistent **art direction** across a clip, including stylized looks. Same story, two styles, and a wide 21:9 banner shape.

In [ ]:
STORY = ("A hiker pitches a green tent with an orange rainfly beside a lake, lights a small stove, and watches the stars come out. "
         "Slow pan from left to right. Audio: gentle music box melody, crickets.")

styles = [fal.run(f"05-style-{name}", "text-to-video", prompt=f"{look}. {STORY}", prompt_expansion_mode="disabled",
                  seed=SEED, duration=5, resolution="768P", aspect_ratio="21:9")
          for name, look in (("claymation", "Stop-motion claymation, visible fingerprints in the clay"),
                             ("pixel-art", "16-bit pixel art, limited palette, crisp pixels"))]
clips += styles
fal.show(*styles)

**What to look for:** does the style hold in **every** frame, or slide back toward photorealism halfway through? Are the tent's colors still recognizable in both styles? Stylized versions are a quick way to make seasonal or playful variants without a new shoot.

<img src="../../.github/images/divider.svg" alt="" width="100%" height="6" />

## 6. Is 1080P worth it, and do we get the length we asked for?

H3 Max generates natively at up to **768P**; **1080P** is a refinement step on top. We render the reference scene from section 2 at 1080P with the same seed. Then we check the docs' note that clips can run about 0.7 s longer than requested.

In [ ]:
weekend_1080 = fal.run("06-reference-1080p", "reference-to-video", prompt=WEEKEND, prompt_expansion_mode="disabled",
                       reference_image_urls=[as_url(IMAGES / f"{name}.webp") for name in CAST],
                       seed=SEED, duration=8, resolution="1080P", aspect_ratio="16:9")
clips.append(weekend_1080)
fal.show(weekend, weekend_1080)

for clip in (weekend, weekend_1080):
    asked, got = clip.record.get("params", {}).get("duration"), clip.info.get("seconds")
    print(f"{clip.name}: asked {asked} s, got {got} s, {clip.info.get('width')}x{clip.info.get('height')}, made in {clip.record.get('seconds')} s")

**What to look for:** sharpen your eyes on small detail (stove knobs, backpack buckles, text on the tent) rather than overall look; refinement adds detail but doesn't change composition. Weigh that against the extra render time. And note the **delivered** duration: if a slot is exactly 8 seconds, plan to trim.

### 🔧 Break it on purpose

Reference-to-video takes up to **9 images** (12 files in total). What happens with 10?

In [ ]:
if MODE == "live":
    ten = [as_url(p) for p in sorted(IMAGES.glob("*.webp"))[:10]]
    try:
        fal.run("break-ten-images", "reference-to-video", prompt="All products from Image 1 to Image 10 on one table.",
                reference_image_urls=ten, prompt_expansion_mode="disabled", duration=5, resolution="480P")
    except RuntimeError as error:
        print(error)
else:
    print("Replay mode: in live mode the service answers 422 for `reference_image_urls`, which allows at most 9 items.")

Validation happens before the job is queued, so nothing renders and nothing is billed. Other limits to keep in mind: reference video and audio clips must each be 2–15 seconds, with 15 seconds combined per type, and the middle frame needs 480P or 768P.

<img src="../../.github/images/divider.svg" alt="" width="100%" height="6" />

## 7. What did each clip cost us in time?

### Scoreboard

Every clip we made, from its saved record: how long the job took end to end, the duration we asked for vs the one we got, and the output's shape and size.

In [ ]:
scoreboard(clips)

**What to look for:** the time `quality` expansion adds, the cost in time of references and of 1080P, and the gap between asked and delivered seconds across all clips. For pricing, use the deployment's page in the Foundry portal. fal's API docs say a clip's price covers the first 4,096 `reference_tokens` (reported in the record), so very large reference sets may cost more.

<img src="../../.github/images/divider.svg" alt="" width="100%" height="6" />

## Teach it to a customer

### What it does, and why it matters

- **Reference-to-video** with up to 12 files (9 images, 3 videos, 3 audio): real products and characters in one generated scene.
- **First, middle, and last frames**, including end-only: land a clip exactly on a brand image.
- **A pinned soundtrack** (`target_audio_url`): picture made to fit our own voiceover or music.
- **Prompt expansion you can read and switch off**, plus a seed for repeatable comparisons.

### A prompt recipe that works

`[Who/what, by reference: "the hiker from Image 1"], [actions in the order they happen], [setting and light]. [Camera move that ends on the money shot]. Audio: [ambience and effects].`

Set `prompt_expansion_mode="disabled"` once the prompt is final, and keep the seed fixed while tuning one setting at a time.

### Honest answers to common questions

- **Is it production-ready?** It's in public preview on Foundry. Validate quality and terms before production.
- **Where does our data go?** To fal, outside Microsoft. Microsoft's product terms, DPA, and certifications don't apply. Review [fal's terms](https://fal.ai/legal) with the customer.
- **Can we use photos of real people?** Only with consent and a policy review. We used generated people only.
- **Why is 1080P slower?** It's refined from a native 768P render, an extra step.
- **Max or Turbo?** Max for references and final assets; [Turbo](../fal-minimax-h3-max-turbo/) for fast drafts with the same text and image modes.

### When something goes wrong

- **422 on references** → more than 9 images, or a video or audio clip outside 2–15 s → trim the list or the clips.
- **422 on `middle_image_url`** → needs both first and last images and 480P or 768P.
- **The clip ignores our wording** → expansion rewrote it; read `expanded_prompt` and set `disabled`.
- **A product drifts** → give it its own reference, or make it the first frame.
- **The clip is longer than the slot** → expect up to ~0.7 s extra and trim in editing.

<img src="../../.github/images/divider.svg" alt="" width="100%" height="6" />

## Your Turn to Explore

- **Ordered references.** Use `reference_conditions` (an ordered list of `{"url": ..., "type": "image"}`) instead of `reference_image_urls` and mix in an audio reference as "Audio 1".
- **Six products.** Add `prop-sleeping-bag.webp` and two catalog photos from `2-scenarios/contoso-retail/data/images/`. When does the scene get crowded?
- **Vertical cut.** Rerun section 2 at `aspect_ratio="9:16"` for a phone story.

In [ ]:
# Your experiment here. Try: fal.run("my-clip", "reference-to-video", prompt="... Image 1 ...", reference_image_urls=[as_url(...)])

<img src="../../.github/images/divider.svg" alt="" width="100%" height="6" />

## What We Did

- ✅ Read what prompt expansion does to our words, with a fixed seed for fair comparisons
- ✅ Put four reference images into one scene and steered clips with first, middle, and last frames
- ✅ Pinned our own voiceover as the soundtrack and held a visual style
- ✅ Checked 768P vs 1080P and requested vs delivered duration, and packaged it for a customer conversation

➡️ **Next:** [MiniMax H3 Max Turbo](../fal-minimax-h3-max-turbo/) runs the same text and image modes faster. Compare the two on one prompt.

Stuck? [Open an issue](https://github.com/microsoft-foundry/model-releases/issues/new) with the failing cell's output.

**Learn more:** [Fal models on Microsoft Foundry](https://learn.microsoft.com/azure/foundry/how-to/fal-ai-models/enable-fal-models) · [Video Generation primer](../../3-primers/video-generation.md) · [MiniMax H3 Max API on fal](https://fal.ai/models/minimax/h3-max/reference-to-video/api)